In [ ]:
# 1. Install necessary libraries silently
!pip install -q transformers datasets evaluate rouge_score accelerate scikit-learn nltk

import warnings
warnings.filterwarnings("ignore", message="Was asked to gather along dimension 0")

import torch
import gc
import os
import numpy as np
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback  
)
import evaluate

# Download the sentence tokenizer model for TF-IDF extraction
nltk.download('punkt')

In [ ]:
# 2. Hardware Detection
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Training on: {device}")

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 3. Load Dataset
print("⏳ Downloading 20,000 training papers from ccdv/arxiv-summarization...")
dataset = load_dataset("ccdv/arxiv-summarization", split="train[:20000]")

print("⏳ Downloading 2000 validation papers...")
val_dataset = load_dataset("ccdv/arxiv-summarization", split="validation[:2000]")

print(f"✅ Success! Loaded {len(dataset)} train and {len(val_dataset)} validation papers.")

In [ ]:
# 4. Tokenizer & Model Initialization
model_checkpoint = "sshleifer/distilbart-cnn-12-6"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint, tie_word_embeddings=False).to(device)

In [ ]:
# 5. TF-IDF Extractive Function
def extract_top_middle_sentences(middle_text, target_word_count=400):
    """Scores sentences using TF-IDF and returns the top ones in chronological order."""
    sentences = nltk.sent_tokenize(middle_text)
    if not sentences:
        return ""

    try:
        vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_matrix = vectorizer.fit_transform(sentences)
        
        sentence_scores = np.array(tfidf_matrix.sum(axis=1)).flatten()
        ranked_indices = sentence_scores.argsort()[::-1]
        
        selected_indices = []
        current_words = 0
        
        for idx in ranked_indices:
            sent_len = len(sentences[idx].split())
            if current_words + sent_len > target_word_count:
                break 
            selected_indices.append(idx)
            current_words += sent_len
            
        selected_indices.sort() # Keep original reading flow
        return " ".join([sentences[i] for i in selected_indices])
        
    except ValueError:
        return " ".join(middle_text.split()[:target_word_count])


In [ ]:
# 6. Smart Truncation Preprocessing (400-400-200 Strategy)
def preprocess_function(examples):
    inputs = []
    
    for doc in examples["article"]:
        doc = doc.replace("\n", " ")
        words = doc.split()
        
        if len(words) > 1000:
            intro = " ".join(words[:400])
            conclusion = " ".join(words[-200:])
            middle_raw = " ".join(words[400:-200])
            middle_extracted = extract_top_middle_sentences(middle_raw, target_word_count=400)
            
            processed_doc = f"{intro} {middle_extracted} {conclusion}"
        else:
            processed_doc = " ".join(words)
            
        inputs.append(processed_doc)

    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)
    labels = tokenizer(text_target=examples["abstract"], max_length=256, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("⏳ Preprocessing text mapping (This will take a few minutes due to TF-IDF)...")
tokenized_train = dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)
print("✅ Preprocessing complete.")


In [ ]:
# 7. Metrics Evaluation
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    if isinstance(predictions, tuple):
        predictions = predictions[0]
        
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}


In [ ]:
# 8. Training Arguments Setup
output_dir = "/kaggle/working/distilbart_arxiv_hybrid"

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,  
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=8, 
    predict_with_generate=True,
    fp16=(device == "cuda"),        
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False 
)

In [ ]:
# 9. Trainer Initialization
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Stops if no improvement for 2 epochs
)

In [ ]:
# 10. Execute Training
print("🚀 Starting Training Loop with Early Stopping...")
trainer.train()

# 11. Save the Final Best Model
trainer.save_model(output_dir)
print(f"🎉 Success! Best model and tokenizer saved locally to: {output_dir}")
